In [1]:
import os 
import random
import numpy as np
import torch 
import torch.nn as nn
import torchvision.models  as models 
import torchvision.transforms.functional as TF
from PIL import Image 

SEED = 42
random.seed(SEED) 
np.random.seed(SEED)
torch.manual_seed(SEED) 

if torch.cuda.is_available(): 
    torch.cuda.manual_seed_all(SEED)
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"--> Execution target confirmed: {device}")
print(f"--> PyTorch Version: {torch.__version__}")

if torch.cuda.is_available(): 
    print(f"---> CUDA Version; {torch.version.cuda}")
    print(f"--> Cloud Hardware Profile: {torch.cuda.get_device_name(0)}")
    
backbone = models.resnet50(weights = models.ResNet50_Weights.DEFAULT)
modules = list(backbone.children())[:-2] 
feature_extractor = nn.Sequential(*modules).to(device)
feature_extractor.eval() 

for param in feature_extractor.parameters(): 
    param.requires_grad = False
    
print("--> Pre-trained ResNet-50 backbone loaded. Classification head pruned sucessfully")

--> Execution target confirmed: cuda
--> PyTorch Version: 2.11.0+cu128
---> CUDA Version; 12.8
--> Cloud Hardware Profile: Tesla T4
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 190MB/s]


--> Pre-trained ResNet-50 backbone loaded. Classification head pruned sucessfully


In [2]:
import torch.nn as nn 
import torch.nn.functional as F 

class SaliencyDecoder(nn.Module): 
    def __init__(self): 
        super(SaliencyDecoder, self).__init__() 
        self.conv1 = nn.Conv2d(2048, 512, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(512, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 1, kernel_size=1)
        
    def forward(self, x): 
        x = F.relu(self.conv1(x)) 
        x = F.interpolate(x, scale_factor=4, mode='bilinear', align_corners=False)
        x = F.relu(self.conv2(x))
        x = F.interpolate(x, scale_factor=4, mode='bilinear', align_corners=False) 
        x = self.conv3(x)
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False) 
        saliency_map = torch.sigmoid(x)
        return saliency_map

decoder = SaliencyDecoder().to(device)

In [3]:
import torch 

def reduce_feature_dimension(features, n_components=48):
    B, C, H, W = features.shape
    N = H * W

    reduced_batches = []

    for b in range(B):
        X = features[b].reshape(C, N)

        X = X - X.mean(dim=1, keepdim=True)

        U, S, Vh = torch.linalg.svd(X, full_matrices=False)

        X_reduced = U[:, :n_components].T @ X

        reduced_batches.append(
            X_reduced.reshape(n_components, H, W)
        )

    return torch.stack(reduced_batches)

def compute_covariance_manifold(features):
    features = reduce_feature_dimension(features, n_components=48)
    B, C, H, W = features.shape
    N = H * W
    
    reshaped_feats = features.reshape(B, C, N) 
    mean_feats = reshaped_feats.mean(dim=2, keepdim=True)
    centered_feats = reshaped_feats - mean_feats
    cov_matrices = torch.bmm(centered_feats, centered_feats.transpose(1, 2)) / (N - 1)
    eps = (
    1e-6
    * torch.eye(C, device=features.device)
    .unsqueeze(0)
    .expand(B, C, C)
)
    spd_matrices = cov_matrices + eps 
    
    return spd_matrices

In [4]:
def matrix_logarithm(spd_matrix): 
    eigenvalues, eigenvectors = torch.linalg.eigh(spd_matrix)
    eigenvalues = torch.clamp(eigenvalues, min=1e-6)
    log_eigenvalues = torch.log(eigenvalues)
    log_diagonal = torch.diag_embed(log_eigenvalues)
    matrix_log = torch.bmm(torch.bmm(eigenvectors, log_diagonal), eigenvectors.transpose(1, 2))
    
    return matrix_log

def riemannian_smoothness_loss(features): 
    spd_mats = compute_covariance_manifold(features)
    tangent = matrix_logarithm(spd_mats)
    tangent_mean = tangent.mean(dim=0, keepdim=True)
    dispersion = ((tangent-tangent_mean)**2).mean()
    
    return dispersion

In [5]:
def compute_cc(pred_map, gt_map): 
     B, C, H, W = pred_map.shape 
     p = pred_map.view(B, -1) 
     g = gt_map.view(B, -1) 
     
     p_mean = p.mean(dim=1, keepdim=True)
     g_mean = g.mean(dim=1, keepdim=True)
     p_centered = p - p_mean 
     g_centered = g - g_mean 
     
     p_norm = torch.norm(p_centered, p=2, dim=1, keepdim=True)
     g_norm = torch.norm(g_centered, p=2, dim=1, keepdim=True) 
     
     eps = 1e-6 
     cc = torch.sum(p_centered * g_centered, dim=1, keepdim=True) / (p_norm * g_norm + eps)
     
     return cc.mean() 

def compute_sim(pred_map, gt_map): 
    B, C, H, W = pred_map.shape 
    p = pred_map.view(B, -1) 
    g = gt_map.view(B, -1) 
    eps = 1e-6 
    
    p_pdf = p / (p.sum(dim=1, keepdim=True) + eps)
    g_pdf = g / (g.sum(dim=1, keepdim=True)+ eps)
    
    sim = torch.sum(torch.minimum(p_pdf, g_pdf), dim=1)
    
    return sim.mean() 

def compute_nss(pred_map, gt_map): 
     B, C, H, W = pred_map.shape 
     pred = pred_map.view(B, -1) 
     gt = gt_map.view(B, -1) 
     
     eps = 1e-6 
     
     pred = (pred - pred.mean(dim=1, keepdim=True)) / (pred.std(dim==1, keepdim=True) + eps)
     fixation_mask = (gt > 0.5).float() 
     n_fix = fixation_mask.sum(dim=1) + eps 
     nss = (pred * fixation_mask).sum(dim=1) / n_fix
     
     return nss.mean() 

In [8]:
import os 
import random 
import torch
from PIL import Image 
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler 
import torchvision.transforms.functional as TF 
from torchvision.transforms import ColorJitter

SEED = 42 
random.seed(SEED) 
torch.manual_seed(SEED)
if torch.cuda.is_available(): 
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

class CAT2000Dataset(Dataset): 
    OFFICIAL_CATEGORIES = [
        "Action", "Affective", "Art", "Cartoon", "DigitalArt", 
        "Everyday", "Fractal", "Indoor", "Inverted", "LightNormal", 
        "LineDrawing", "Loud", "LowResolution", "Natural", "Object",
        "Outdoor", "Photo", "Random", "Social", "Synthetic" 
    ]
    
    def __init__(self, root_dir, augment=False, test_mode=False): 
        self.root_dir = os.path.abspath(root_dir)
        self.stimuli_dir = os.path.join(root_dir, "Stimuli")
        self.maps_dir = os.path.join(root_dir, "FixationMaps")
        self.test_mode = test_mode

        if test_mode:
            self.maps_dir = None
        else:
            self.maps_dir = os.path.join(root_dir, "FixationMaps")
        self.augment = augment 
        self.color_jitter = ColorJitter( 
            brightness = 0.15, 
            contrast = 0.15,  
            saturation = 0.10, 
            hue = 0.02
                                        )
        self.cat_to_idx = { 
            cat: idx 
            for idx, cat in enumerate(self.OFFICIAL_CATEGORIES)
            }
        self.samples = [] 
        self._index_dataset() 
        
    def _index_dataset(self): 
        for cat in self.OFFICIAL_CATEGORIES: 
            
            stim_folder = os.path.join(self.stimuli_dir, cat)
            
            if not os.path.exists(stim_folder): 
                continue
            
            if self.test_mode: 
                map_folder = os.path.join(stim_folder, "Output")
            else: 
                map_folder = os.path.join(self.maps_dir, cat)
            
            for filename in sorted(os.listdir(stim_folder)): 
                if not filename.lower().endswith((".jpg", ".jpeg", ".png")): 
                    continue
                
                image_path = os.path.join(stim_folder, filename) 
                
                if self.test_mode:
                    stem = os.path.splitext(filename)[0]
                    map_path = os.path.join(
                        map_folder,
                        f"{stem}_SaliencyMap.jpg"
                    )
                else:
                    map_path = os.path.join(map_folder, filename)

                if os.path.exists(map_path):
                    self.samples.append({
                        "image": image_path,
                        "map": map_path,
                        "cat_idx": self.cat_to_idx[cat]
                    })
        
    def __len__(self): 
        return len(self.samples)
    
    def __getitem__(self, idx): 
        sample = self.samples[idx] 
        image = Image.open(sample["image"]).convert("RGB")
        fix_map = Image.open(sample["map"]).convert("L")
        
        if self.augment: 
            if random.random() < 0.5: 
                image = TF.hflip(image)
                fix_map = TF.hflip(fix_map)
                
            image = self.color_jitter(image)
        
        image = TF.resize(
            image,
            (224, 224),
            interpolation=TF.InterpolationMode.BILINEAR
        )
        fix_map = TF.resize( 
            fix_map, 
            (224, 224), 
            interpolation=TF.InterpolationMode.BILINEAR
            )
        
        image_tensor = TF.to_tensor(image)
        fix_tensor = TF.to_tensor(fix_map)
        
        fix_tensor = (
        fix_tensor - fix_tensor.min()
        ) / (
        fix_tensor.max() - fix_tensor.min() + 1e-6
        )
    
        image_tensor = TF.normalize( 
                image_tensor, 
                mean=[0.485, 0.456, 0.406], 
                std = [0.229, 0.224, 0.225])
    
        return image_tensor, fix_tensor, sample["cat_idx"]
    
def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    torch.manual_seed(worker_seed)
    
base_path = "/content/drive/MyDrive/CAT2000/trainSet"
full_dataset = CAT2000Dataset( 
            root_dir=base_path, 
            augment=True
            )
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size 

generator = torch.Generator().manual_seed(SEED)

train_dataset, val_dataset = random_split( 
                full_dataset,  
                [train_size, val_size], 
                generator= generator
                )
train_labels = [
    full_dataset.samples[i]["cat_idx"]
    for i in train_dataset.indices
]
class_counts = torch.bincount(torch.tensor(train_labels, dtype=torch.long))
class_weights = 1.0 / class_counts.float() 

sample_weights = [ 
        class_weights[label] 
        for label in train_labels
]
sampler = WeightedRandomSampler( 
            sample_weights, 
            num_samples = len(sample_weights),
            replacement=True
        )

production_loader = DataLoader( 
        train_dataset, 
        batch_size=8, 
        sampler= sampler,
        worker_init_fn=seed_worker
    )
validation_loader = DataLoader( 
        val_dataset, 
        batch_size=8, 
        shuffle=False,
        worker_init_fn=seed_worker
)

print(f"Total images      : {len(full_dataset)}")
print(f"Training images   : {len(train_dataset)}")
print(f"Validation images : {len(val_dataset)}")
print(f"Categories        : {len(full_dataset.OFFICIAL_CATEGORIES)}")
print(f"Random seed      : {SEED}")


Total images      : 1200
Training images   : 960
Validation images : 240
Categories        : 20
Random seed      : 42


In [8]:
import torch.optim as optim
import numpy as np

optimizer = optim.Adam(decoder.parameters(), lr=1e-4)

NUM_EPOCHS = 10
ALPHA = 0.05

best_cc = -float("inf")

train_history = {
    "loss": [],
    "cc": [],
    "sim": [],
    "smoothness": [],
    "val_loss": [],
    "val_cc": [],
    "val_sim": []
}

print("--> Starting training cycle...")

for epoch in range(NUM_EPOCHS):

    decoder.train()

    epoch_total_loss = 0.0
    epoch_cc = 0.0
    epoch_sim = 0.0
    epoch_smooth = 0.0

    for images, fixation_maps, _ in production_loader:

        images = images.to(device)
        fixation_maps = fixation_maps.to(device)

        optimizer.zero_grad()

        with torch.no_grad():
            features = feature_extractor(images)

        predicted_saliency = decoder(features)

        cc = compute_cc(predicted_saliency, fixation_maps)
        sim = compute_sim(predicted_saliency, fixation_maps)
        smoothness = riemannian_smoothness_loss(features)

        cc_loss = 1 - cc
        total_loss = cc_loss + ALPHA * smoothness

        total_loss.backward()
        optimizer.step()

        epoch_total_loss += total_loss.item()
        epoch_cc += cc.item()
        epoch_sim += sim.item()
        epoch_smooth += smoothness.item()

    avg_loss = epoch_total_loss / len(production_loader)
    avg_cc = epoch_cc / len(production_loader)
    avg_sim = epoch_sim / len(production_loader)
    avg_smooth = epoch_smooth / len(production_loader)


    decoder.eval()

    val_loss = 0.0
    val_cc = 0.0
    val_sim = 0.0

    with torch.no_grad():

        for images, fixation_maps, _ in validation_loader:

            images = images.to(device)
            fixation_maps = fixation_maps.to(device)

            features = feature_extractor(images)
            predicted_saliency = decoder(features)

            cc = compute_cc(predicted_saliency, fixation_maps)
            sim = compute_sim(predicted_saliency, fixation_maps)
            smoothness = riemannian_smoothness_loss(features)

            loss = (1 - cc) + ALPHA * smoothness

            val_loss += loss.item()
            val_cc += cc.item()
            val_sim += sim.item()

    avg_val_loss = val_loss / len(validation_loader)
    avg_val_cc = val_cc / len(validation_loader)
    avg_val_sim = val_sim / len(validation_loader)


    train_history["loss"].append(avg_loss)
    train_history["cc"].append(avg_cc)
    train_history["sim"].append(avg_sim)
    train_history["smoothness"].append(avg_smooth)

    train_history["val_loss"].append(avg_val_loss)
    train_history["val_cc"].append(avg_val_cc)
    train_history["val_sim"].append(avg_val_sim)


    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {avg_loss:.4f} | "
        f"Train CC: {avg_cc:.4f} | "
        f"Train SIM: {avg_sim:.4f} | "
        f"Smoothness: {avg_smooth:.6f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val CC: {avg_val_cc:.4f} | "
        f"Val SIM: {avg_val_sim:.4f}"
    )

    if avg_val_cc > best_cc:

        best_cc = avg_val_cc

        torch.save(
            decoder.state_dict(),
            "/content/drive/MyDrive/CAT2000/best_decoder.pth"
        )

np.save(
    "/content/drive/MyDrive/CAT2000/train_history.npy",
    train_history
)

print("\n--> Training Complete.")
print(f"Best Validation CC: {best_cc:.4f}")

--> Starting training cycle...
Epoch [1/10] | Train Loss: 0.3408 | Train CC: 0.6593 | Train SIM: 0.3631 | Smoothness: 0.002126 | Val Loss: 0.3520 | Val CC: 0.6481 | Val SIM: 0.3515
Epoch [2/10] | Train Loss: 0.2560 | Train CC: 0.7441 | Train SIM: 0.3579 | Smoothness: 0.002126 | Val Loss: 0.3258 | Val CC: 0.6743 | Val SIM: 0.3489
Epoch [3/10] | Train Loss: 0.2152 | Train CC: 0.7849 | Train SIM: 0.3508 | Smoothness: 0.002111 | Val Loss: 0.3142 | Val CC: 0.6859 | Val SIM: 0.3442
Epoch [4/10] | Train Loss: 0.1850 | Train CC: 0.8151 | Train SIM: 0.3475 | Smoothness: 0.002109 | Val Loss: 0.3195 | Val CC: 0.6806 | Val SIM: 0.3411
Epoch [5/10] | Train Loss: 0.1681 | Train CC: 0.8320 | Train SIM: 0.3494 | Smoothness: 0.002108 | Val Loss: 0.3085 | Val CC: 0.6917 | Val SIM: 0.3425
Epoch [6/10] | Train Loss: 0.1468 | Train CC: 0.8533 | Train SIM: 0.3442 | Smoothness: 0.002110 | Val Loss: 0.2945 | Val CC: 0.7056 | Val SIM: 0.3415
Epoch [7/10] | Train Loss: 0.1359 | Train CC: 0.8642 | Train SIM: 0.3

In [33]:
import numpy as np
import torch

torch.save(
    decoder.state_dict(),
    "/content/drive/MyDrive/CAT2000/final_decoder.pth"
)

print("Final model saved successfully.")
print("Training history saved successfully.")
print("Best model is stored separately as best_decoder.pth")

Final model saved successfully.
Training history saved successfully.
Best model is stored separately as best_decoder.pth


In [9]:
analysis_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    worker_init_fn=seed_worker,
    pin_memory=torch.cuda.is_available()
)

test_dataset = CAT2000Dataset(
    root_dir="/content/drive/MyDrive/CAT2000/testSet/",
    augment=False,
    test_mode=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    worker_init_fn=seed_worker,
    pin_memory=torch.cuda.is_available()
)

print(f"Validation images : {len(val_dataset)}")
print(f"Test images       : {len(test_dataset)}")

Validation images : 240
Test images       : 1200


In [13]:
import torch 

decoder.load_state_dict( 
        torch.load("/content/drive/MyDrive/CAT2000/best_decoder.pth", 
                   map_location=device
            )
        )

decoder.to(device)
decoder.eval() 
feature_extractor.eval() 

all_images = []
all_features = []
all_predictions = []
all_targets = []
all_categories = []

print("Running inference on validation set")

with torch.no_grad(): 
    for images, fixation_maps, categories in analysis_loader: 
        images = images.to(device)
        fixation_maps = fixation_maps.to(device)
        features = feature_extractor(images)
        predictions = decoder(features)
        all_features.append(features.cpu()) 
        all_predictions.append(predictions.cpu()) 
        all_targets.append(fixation_maps.cpu()) 
        all_categories.append(categories) 
        all_images.append(images.cpu())
        
all_features = torch.cat(all_features, dim=0) 
all_predictions = torch.cat(all_predictions, dim=0) 
all_targets = torch.cat(all_targets, dim=0) 
all_categories = torch.cat(all_categories, dim=0) 
all_images = torch.cat(all_images, dim=0)

print("\nInference complete.\n")

print(f"Features     : {all_features.shape}")
print(f"Predictions  : {all_predictions.shape}")
print(f"Ground Truth : {all_targets.shape}")
print(f"Categories   : {all_categories.shape}")

Running inference on validation set

Inference complete.

Features     : torch.Size([240, 2048, 7, 7])
Predictions  : torch.Size([240, 1, 224, 224])
Ground Truth : torch.Size([240, 1, 224, 224])
Categories   : torch.Size([240])


In [10]:
import os 

analysis_dir = "\content\drive\MyDrive\CAT2000\analysis"
os.makedirs(analysis_dir, exist_ok=True)

torch.save(all_features, os.path.join(analysis_dir, "features.pt"))
torch.save(all_predictions, os.path.join(analysis_dir, "predictions.pt"))
torch.save(all_targets, os.path.join(analysis_dir, "targets.pt"))
torch.save(all_categories, os.path.join(analysis_dir, "categories.pt"))

print("Analysis tensors saved successfully.")

<>:3: SyntaxWarning: invalid escape sequence '\c'
<>:3: SyntaxWarning: invalid escape sequence '\c'
/tmp/ipykernel_37588/2275874249.py:3: SyntaxWarning: invalid escape sequence '\c'
  analysis_dir = "\content\drive\MyDrive\CAT2000\analysis"


Analysis tensors saved successfully.


In [15]:
import torch
import numpy as np 

def compute_channel_entropy(features): 
    B, C, H, W = features.shape 
    entropy = [] 
    for i in range(B): 
        
        feat = features[i] 
        channel_energy = feat.abs().mean(dim=(1,2))
        probs = channel_energy / (channel_energy.sum() + 1e-12)
        H_channel = -(probs * torch.log2(probs + 1e-12)).sum() 
        entropy.append(H_channel.item())
        
    return np.array(entropy)

def compute_spatial_correlation(features):

    features = reduce_feature_dimension(features, n_components=48)

    B, C, H, W = features.shape
    correlations = []

    for i in range(B):

        feat = features[i].reshape(C, -1)
        feat = feat - feat.mean(dim=1, keepdim=True)
        std = feat.std(dim=1, keepdim=True)
        valid = (std.squeeze() > 1e-6)
        feat = feat[valid]
        if feat.shape[0] < 2:
            correlations.append(0.0)
            continue
        feat = feat / (feat.std(dim=1, keepdim=True) + 1e-8)
        corr = feat @ feat.T
        corr /= (feat.shape[1] - 1)
        mask = ~torch.eye(
            corr.shape[0],
            dtype=torch.bool,
            device=corr.device
        )

        correlations.append(
            corr[mask].abs().mean().item()
        )

    return np.array(correlations)

def compute_manifold_smoothness(features):

    spd = compute_covariance_manifold(features)

    tangent = matrix_logarithm(spd)

    tangent_mean = tangent.mean(dim=0, keepdim=True)

    smoothness = (
        (tangent - tangent_mean) ** 2
    ).mean(dim=(1,2))

    return smoothness.cpu().numpy()

def analyze_feature_tensor(features):

    results = {}

    results["entropy"] = compute_channel_entropy(features)

    results["spatial_corr"] = compute_spatial_correlation(features)

    results["smoothness"] = compute_manifold_smoothness(features)

    return results

In [16]:
import os
import numpy as np

results = analyze_feature_tensor(all_features)

print("Analysis complete.\n")

for metric, values in results.items():
    print(
        f"{metric:15s}"
        f"Mean = {values.mean():.6f}   "
        f"Std = {values.std():.6f}"
    )

analysis_dir = "/content/drive/MyDrive/CAT2000/analysis"

np.save(
    os.path.join(analysis_dir, "pretrained_analysis.npy"),
    results,
    allow_pickle=True
)

print("\nAnalysis saved successfully.")

Analysis complete.

entropy        Mean = 8.545672   Std = 0.636382
spatial_corr   Mean = 0.000721   Std = 0.001959
smoothness     Mean = 0.025043   Std = 0.027461

Analysis saved successfully.
